# Sherrington-Kirkpatrick Model

The energy of the Sherrington-Kirkpatrick (SK) model is
$$
E = \frac{1}{\sqrt{N}}\sum_{i<j\leq N} J_{ij} \sigma_i \sigma_j,
$$
where the couplings $J_{ij}$ are independent standard normal numbers. The QAOA looks for the spins $\sigma_i = \pm 1$ of lowest energy.

In [ ]:
using QAOA, Random
import MFAOA

using PyPlot

## QAOA

__Defining the problem by hand:__

In [ ]:
N = 4
rng = Xoshiro(1)
J = zeros(N, N)
for j in 1:N, i in 1:j-1
    J[i, j] = J[j, i] = randn(rng) / sqrt(N)
end
SK_problem = IsingProblem(zeros(N), J)    # minimized by default

__Using the wrapper function:__ it draws the same couplings.

In [ ]:
SK_problem = sherrington_kirkpatrick(N; seed = 1)
SK_problem.J == J

__Optimizing the angles:__ `optimize_schedule` tunes the angles of `p` layers and returns them with the average energy they give:

In [ ]:
p = 2
angles, energy = optimize_schedule(SK_problem, p)

In [ ]:
probs = probabilities(SK_problem, angles)

xlabels = []
for bstr in digits.(0:2^N-1, base=2, pad=N)
    push!(xlabels, "\$|" * prod([string(b) for b in bstr]) * "\\rangle\$")
end

figure(figsize=(5, 3.2))
ax = subplot(111)
bar(0:2^N-1, probs)
ax.set_xticks(0:2^N-1)
ax.set_xticklabels(xlabels, rotation=90)
minorticks_off()
tight_layout()

## Mean-Field AOA with MFAOA.jl

The [mean-field approximate optimization algorithm](https://doi.org/10.1103/PRXQuantum.4.030335) (mean-field AOA) replaces every qubit by a classical spin vector $\mathbf{n}_i = (n_i^x, n_i^y, n_i^z)$ and runs a QAOA-like schedule on these spins. The package [MFAOA.jl](https://github.com/Quicopt/MFAOA.jl) implements it. It describes problems and angles in the same way as QAOA.jl and exports many of the same names, so we load it with `import`.

The SK energy has no local fields, so flipping every spin leaves it unchanged. The mean-field AOA cannot break this symmetry by itself, so we fix the last spin to $\sigma_N = +1$. Its couplings to the other spins then act on them as local fields $h_i = J_{iN}$:

In [ ]:
h_fixed = J[1:N-1, N]        # local fields from the fixed spin σ_N = +1
J_fixed = J[1:N-1, 1:N-1]    # couplings among the remaining spins
mf_problem = MFAOA.IsingProblem(h_fixed, J_fixed)    # minimized by default, as in QAOA.jl

__Following the spins through the schedule:__ we take MFAOA's linear schedule with $p = 1000$ steps and record every spin vector after each step.

In [ ]:
p = 1000
mf_schedule = MFAOA.linear_schedule(p, 0.5)

workspace = MFAOA.Workspace(mf_problem)
spins = workspace.state                 # spins.x[i], spins.y[i], spins.z[i] are the components of n_i
history = [[spins.x spins.y spins.z]]   # history[k][i, :] is n_i after k - 1 steps
for k in 1:p
    MFAOA.step!(workspace, mf_problem, mf_schedule.β[k], mf_schedule.γ[k])
    push!(history, [spins.x spins.y spins.z])
end

In [ ]:
# plot x, y, and z of all spins that are not fixed
times = range(0, 1, p + 1)

figure(figsize=((N - 1) * 2.2, 2))
for i in 1:N-1
    subplot(1, N - 1, i)
    for (component, label) in enumerate(["x", "y", "z"])
        plot(times, [n[i, component] for n in history], label=label)
    end
    xlim(0, 1)
    ylim(-1, 1)
    xlabel("t/T")
    ylabel("n_" * string(i))
    i == 1 && legend(fontsize=7)
end
tight_layout()

__Comparing the solutions:__ the sign of $n_i^z$ at the end of the schedule is the spin $\sigma_i$. `MFAOA.solve` runs the same schedule and returns these spins with their energy. For $N = 4$ spins, we can also find the lowest energy by trying all $2^N = 16$ spin configurations:

In [ ]:
result = MFAOA.solve(mf_problem, mf_schedule)
configurations = [[(-1)^b for b in digits(s, base=2, pad=N)] for s in 0:2^N-1]

println("QAOA, expectation value:   ", energy)
println("mean-field AOA solution:   ", result.objective)
println("lowest over all solutions: ", minimum(σ -> objective(SK_problem, σ), configurations))

The mean-field AOA finds the lowest energy. The QAOA value is an average over all measured assignments, so it stays above the lowest energy.

## Annealing

With many layers and slowly changing angles, the QAOA becomes a quantum annealing. `linear_schedule(p, τ)` cuts an annealing of duration $p\tau$ into $p$ layers:

In [ ]:
p = 256
τ = 8 / p
probs = probabilities(SK_problem, linear_schedule(p, τ))